In [ ]:
# Install necessary packages
!pip install sacrebleu spacy xlsxwriter openpyxl
!python -m spacy download es_core_news_sm
!python -m spacy download en_core_web_sm

import pandas as pd
import os
import spacy
import sacrebleu
from openpyxl import load_workbook
import shutil
from spacy.symbols import ORTH, POS
from spacy.language import Language

# Load the Spanish and English language models
nlp_es = spacy.load('es_core_news_sm')
nlp_en = spacy.load('en_core_web_sm')

# Custom function to add multi-word entities and specific cases to the tokenizer
def create_custom_tokenizer(nlp):
    # Define the phrases and tokens to be treated as single tokens or with specific POS tags
    special_cases = [
        ("DDR",),  # Ensure DDR is recognized as a single token
        ("«",),    # Ensure « is recognized as punctuation
        ("»",)     # Ensure » is recognized as punctuation
    ]

    # Add special cases to the tokenizer
    for case in special_cases:
        nlp.tokenizer.add_special_case(case[0], [{ORTH: case[0]}])

    # Set specific POS for "DDR" as PROPN
    nlp.get_pipe("ner").add_label("PROPN")  # Ensure "PROPN" is a recognized label in NER

# Register the custom POS tagger as a component
@Language.component("custom_pos_tagger")
def custom_pos_tagger(doc):
    for token in doc:
        if token.text == "DDR":
            token.pos_ = "PROPN"  # Set DDR as PROPN
        elif token.text in ["«", "»"]:
            token.pos_ = "PUNCT"  # Set « and » as PUNCT
    return doc

# Create custom tokenizers for both languages and add the custom POS tagger
create_custom_tokenizer(nlp_en)
create_custom_tokenizer(nlp_es)
nlp_en.add_pipe("custom_pos_tagger", last=True)
nlp_es.add_pipe("custom_pos_tagger", last=True)

# Function to process Excel files in the "texts" folder
def process_excel_files(input_folder_path, output_folder_path):
    try:
        # Create the output directory if it doesn't exist
        os.makedirs(output_folder_path, exist_ok=True)

        for file_name in os.listdir(input_folder_path):
            if file_name.endswith('.xlsx'):
                file_path = os.path.join(input_folder_path, file_name)
                print(f"Processing file: {file_path}")

                # Load the workbook and each sheet
                workbook = load_workbook(file_path)
                sheet_names = workbook.sheetnames

                for sheet_name in sheet_names:
                    worksheet = workbook[sheet_name]

                    # Create a DataFrame from the worksheet values
                    data = worksheet.values
                    columns = next(data)[0:]  # The first row is the header
                    df = pd.DataFrame(data, columns=columns)

                    # Check if required columns exist
                    if 'MT RESULT' in df.columns and 'TRANSLATOR RESULT' in df.columns and 'SOURCE TEXT' in df.columns and 'TIME SPENT' in df.columns:
                        # Define columns to process
                        mt_column = 'MT RESULT'
                        translator_column = 'TRANSLATOR RESULT'
                        source_column = 'SOURCE TEXT'
                        time_spent_column = 'TIME SPENT'

                        # Calculate TER for each row and add it to a new column
                        def safe_calculate_ter(mt_result, translator_result):
                            if isinstance(mt_result, str) and isinstance(translator_result, str):
                                return calculate_ter(mt_result, translator_result)
                            return None

                        df['TER'] = df.apply(lambda row: safe_calculate_ter(row[mt_column], row[translator_column]), axis=1)

                        # Extract lemmas and POS tags for each text segment in specified columns
                        df['lemmas_source'], df['pos_source'] = zip(*df[source_column].apply(lambda x: extract_lemmas_pos(x, nlp_en) if isinstance(x, str) else ([], [])))
                        df['lemmas_mt'], df['pos_mt'] = zip(*df[mt_column].apply(lambda x: extract_lemmas_pos(x, nlp_es) if isinstance(x, str) else ([], [])))
                        df['lemmas_trans'], df['pos_trans'] = zip(*df[translator_column].apply(lambda x: extract_lemmas_pos(x, nlp_es) if isinstance(x, str) else ([], [])))

                        # Add column to count content words in 'pos_trans'
                        def count_content_words(pos_tags):
                            content_words = {'NOUN', 'PROPN', 'VERB', 'ADJ', 'ADV'}
                            return sum(tag in content_words for tag in pos_tags)

                        df['total content words pos_trans'] = df['pos_trans'].apply(count_content_words)

                        # Reorder columns to place TER after TIME SPENT and other columns after TER
                        ter_index = df.columns.get_loc(time_spent_column) + 1
                        df.insert(loc=ter_index, column='TER', value=df.pop('TER'))

                        lemmas_pos_columns = {
                            'lemmas_source': 'lemmas_source',
                            'pos_source': 'pos_source',
                            'lemmas_mt': 'lemmas_mt',
                            'pos_mt': 'pos_mt',
                            'lemmas_trans': 'lemmas_trans',
                            'pos_trans': 'pos_trans',
                            'total content words pos_trans': 'total content words pos_trans',
                        }

                        for col_name in lemmas_pos_columns:
                            df.insert(loc=ter_index + 1, column=col_name, value=df.pop(col_name))
                            ter_index += 1

                        # Save the processed DataFrame to a new Excel file in the output folder
                        output_file_path = os.path.join(output_folder_path, f"processed_{file_name}")

                        with pd.ExcelWriter(output_file_path, engine='openpyxl') as writer:
                            df.to_excel(writer, sheet_name=sheet_name, index=False)

                        print(f"Processed data saved to: {output_file_path}")
                    else:
                        print(f"Skipped file {file_name} in sheet {sheet_name}: Missing required columns.")

    except FileNotFoundError:
        print(f"Error: The folder '{input_folder_path}' does not exist or cannot be accessed.")

# TER calculation using sacreBLEU, with normalization if the score is greater than 1
def calculate_ter(mt_result, translator_result):
    # Calculate TER using sacrebleu for individual sentences
    ter_metric = sacrebleu.metrics.TER()
    ter_score = ter_metric.sentence_score(mt_result, [translator_result]).score

    # Normalize only if the value is above 1
    if ter_score > 1:
        ter_score_normalized = ter_score / 100
    else:
        ter_score_normalized = ter_score  # If already normalized, do not divide

    return min(ter_score_normalized, 1.0)  # Limit maximum value to 1

def extract_lemmas_pos(text, nlp_model):
    # Process text using the specified SpaCy model
    doc = nlp_model(text)
    # Extract lemmas and POS tags
    lemmas = [token.lemma_ for token in doc]
    pos_tags = [token.pos_ for token in doc]
    return lemmas, pos_tags

# Define the path to the 'texts' folder and the 'texts_processed' folder in Google Colab
colab_texts_folder = '/content/texts'
colab_texts_processed_folder = '/content/texts_processed'

# Process Excel files in the 'texts' folder and save them in the 'texts_processed' folder
process_excel_files(colab_texts_folder, colab_texts_processed_folder)

# Step to create a zip file containing all processed files
output_zip_path = '/content/processed_texts.zip'
shutil.make_archive('/content/processed_texts', 'zip', colab_texts_processed_folder)

print(f"All processed files have been zipped into: {output_zip_path}")

